# CP1 Week 14: Complete sensor-log project — Complete Worked Solutions

For students who have attempted the matching weekly notebook. Read the problem,
explain the steps aloud, then compare your code with this reference. Other correct
implementations are possible. These are private self-study examples, not submitted grades.

Run cells from top to bottom in a fresh Python 3 notebook. Every numbered exercise
and the week's bridge (when present) has a worked solution and representative checks.
`assert condition` raises an error if a check fails; a passing check produces no output.
Assertions here verify selected cases, not every possible input.

**Prerequisites:** functions, loops, lists, strings, dictionaries and returned tuples
(introduced in Week 11). File tasks create their own small sample files in the runtime's
current working directory. No live keyboard input or external downloads are needed.

**Türkçe:** Önce kendiniz deneyin; sonra adımları ve sınır durumlarını karşılaştırın.
Doğru sonuç kadar, neden o sonucu aldığınızı açıklamak da önemlidir.


## Reading the worked solutions

Each section gives the problem, a plan, Python code, and an expected result or checks.
Try your own answer first. After comparing, close this notebook and solve a changed example.

An `assert condition` line checks that a result matches an expectation. If the condition is false,
Python raises `AssertionError`; it means that check failed. For decimal calculations, a check such
as `abs(actual - expected) < 0.001` allows a small rounding difference.

Some examples run several small test cases. A pair `(input_value, expected_result)` keeps an input
beside its expected answer. In `for value, expected in cases`, Python takes the two parts of each
pair and gives them those names. Follow the calculation inside the loop for one case at a time.

**Türkçe:** `assert`, beklenen sonuç ile gerçek sonucu karşılaştırır. Test başarısızsa önce girdiyi
ve ara değerleri incele. Birden fazla örnek varsa önce yalnızca bir örneğin adımlarını takip et.

File examples create sample files in the current runtime folder. Use a separate runtime for your
own practice so the worked answers do not supply hidden variables to it.

[Week 14 lesson](../notebooks/Week_14.ipynb) · [All solutions](README.md) · [Course guide](../STUDY_GUIDE.md)


### Small library reference for the sensor project

These tools handle file details while your functions express the processing steps.
All belong to Python's standard library; no extra installation is needed.

| Tool | Read it as | Why it appears here |
|---|---|---|
| `csv.reader(file)` | Read one CSV record as a list of text fields | Handles CSV field boundaries, including quoted fields |
| `csv.writer(file).writerows(rows)` | Write a list of records to CSV | Saves the same three-column structure |
| `math.isfinite(value)` | Is this a finite real number? | Rejects `nan`, `inf` and `-inf` after conversion |
| `datetime.strptime(text, format)` | Interpret text using an expected date/time pattern | Rejects invalid dates; `%Y-%m-%d %H:%M` means year-month-day hour:minute |
| `date.strftime(format)` | Turn the date/time back into text | Lets us compare with the required written format |
| `Path(folder) / filename` | Join a folder and filename | Here `/` joins paths; it is not numeric division |
| `path.read_text(encoding="utf-8")` | Read the file's text | Lets a check inspect the saved report |
| `TemporaryDirectory()` | Create a temporary folder for test files | Keeps test fixtures separate from the reference output files |

**Türkçe:** Bu yardımcılar dosya ve biçim ayrıntılarını yönetir. Asıl algoritma yine aynıdır: oku, doğrula, hesapla ve kaydet. `Path` ile kullanılan `/` işareti klasör ile dosya adını birleştirir.


### Pipeline map and core milestones

EX1 creates the supplied data → EX2 reads it → EX3 validates one row → EX4 cleans all
rows → EX5–6 calculate statistics → EX7 builds a report → EX8–9 save both files →
EX10 integrates and checks the complete tool. These ten stages are core.
EX11–12 are optional extensions. Functions are introduced before their first call.

The file schema is `timestamp,sensor,value`. Valid ranges are inclusive: temperature
−50…60°C, humidity 0…100%, pressure 800…1200 hPa. Empty sensor groups have no mean:
store `None`, display `N/A`. A wrong header or missing file is a file-level error;
an invalid record is rejected individually so other records can still be processed.


## Exercise 1: Create the supplied sample data

Run the supplied dataset setup, producing `sensor_data.csv` with columns
`timestamp,sensor,value`. This is an execution milestone, not a missing-code task.
The full 20-record dataset is included below so this solution runs independently.


### Worked reasoning

This supplied setup stage creates the exact reference dataset. Do not correct bad rows by hand: the later validation stage must identify them. The 20 data rows include two empty readings, one non-numeric reading and two out-of-range readings. Use Python’s standard csv module to handle CSV field boundaries; no package installation is needed.

**Türkçe:** Bozuk satırları elle düzeltmeyin; temizleme algoritmasının bunları bulmasını istiyoruz.

**Expected check:** One header plus 20 records; the data is unchanged from the student notebook.


In [1]:
import csv
import math
from datetime import datetime
from pathlib import Path

SENSOR_RANGES = {"temp": (-50.0, 60.0), "humidity": (0.0, 100.0), "pressure": (800.0, 1200.0)}
TIME_FORMAT = "%Y-%m-%d %H:%M"
HEADER = ["timestamp", "sensor", "value"]

sample_data = """timestamp,sensor,value
2024-01-15 08:00,temp,22.5
2024-01-15 08:05,humidity,45.2
2024-01-15 08:10,temp,23.1
2024-01-15 08:15,pressure,1013.25
2024-01-15 08:20,temp,-999
2024-01-15 08:25,humidity,
2024-01-15 08:30,temp,22.8
2024-01-15 08:35,humidity,46.1
2024-01-15 08:40,pressure,abc
2024-01-15 08:45,temp,23.5
2024-01-15 08:50,humidity,44.8
2024-01-15 08:55,pressure,1012.80
2024-01-15 09:00,temp,24.0
2024-01-15 09:05,humidity,150.0
2024-01-15 09:10,pressure,1013.50
2024-01-15 09:15,temp,22.2
2024-01-15 09:20,humidity,43.5
2024-01-15 09:25,pressure,1011.90
2024-01-15 09:30,temp,
2024-01-15 09:35,humidity,47.3"""

with open("sensor_data.csv", "w", encoding="utf-8", newline="") as file:
    file.write(sample_data + "\n")
print("sensor_data.csv created: 20 data records plus a header")
with open("sensor_data.csv", "r", encoding="utf-8", newline="") as file:
    saved_rows = list(csv.reader(file))
assert saved_rows[0] == HEADER and len(saved_rows) == 21


sensor_data.csv created: 20 data records plus a header


## Exercise 2: Read and Parse the CSV

Write code that reads `"sensor_data.csv"` and stores all data rows in a list called `raw_data`. Each element should be a list of 3 strings: `[timestamp, sensor, value]`.

**Expected output:**
```
Header: ['timestamp', 'sensor', 'value']
Total rows read: 20
First row: ['2024-01-15 08:00', 'temp', '22.5']
Last row: ['2024-01-15 09:35', 'humidity', '47.3']
```

### Worked reasoning

Open the file with explicit encoding and CSV newline handling. Read the header once, validate its three names, then collect data rows. An empty file returns an empty list; a wrong nonempty header raises ValueError. csv.reader returns fields as strings, which is exactly what the row validator needs next.

**Türkçe:** CSV okuma alanları ayırır; geçerlilik kontrolü sonraki aşamadır. Boş dosya ile yanlış başlık farklı durumlardır.

**Expected check:** 20 rows; first temp22.5 at08:00, last humidity47.3 at09:35.


In [2]:
def read_data(filename):
    with open(filename, "r", encoding="utf-8", newline="") as file:
        reader = csv.reader(file)
        header = next(reader, None)
        if header is None:
            return []
        if [field.strip() for field in header] != HEADER:
            raise ValueError("expected header: timestamp,sensor,value")
        return list(reader)

raw_data = read_data("sensor_data.csv")
print("Header:", HEADER)
print("Total rows read:", len(raw_data))
print("First row:", raw_data[0])
print("Last row:", raw_data[-1])
assert len(raw_data) == 20
assert raw_data[0] == ["2024-01-15 08:00", "temp", "22.5"]
assert raw_data[-1] == ["2024-01-15 09:35", "humidity", "47.3"]


Header: ['timestamp', 'sensor', 'value']
Total rows read: 20
First row: ['2024-01-15 08:00', 'temp', '22.5']
Last row: ['2024-01-15 09:35', 'humidity', '47.3']


## Exercise 3: Validate a Single Row

Write a function called `validate_row(row)` that takes a list `[timestamp, sensor, value_string]` and returns a tuple `(is_valid, reason)`.

- If the row is valid: return `(True, "OK")`
- If the value is empty: return `(False, "empty value")`
- If the value is not a number: return `(False, "non-numeric value: 'xxx'")`
- If the value is out of range: return `(False, "out of range: xxx")`

**Expected output (test cases):**
```
['2024-01-15 08:00', 'temp', '22.5']     -> (True, 'OK')
['2024-01-15 08:25', 'humidity', '']      -> (False, 'empty value')
['2024-01-15 08:40', 'pressure', 'abc']   -> (False, "non-numeric value: 'abc'")
['2024-01-15 08:20', 'temp', '-999']      -> (False, 'out of range: -999.0')
['2024-01-15 09:05', 'humidity', '150.0'] -> (False, 'out of range: 150.0')
```



**Before indexing:** first check the row has exactly three string fields.
Then check required text, sensor membership, and timestamp format. Convert the value
with `float()`, catch conversion failures, test `math.isfinite(value)`, then test the
inclusive sensor range. `datetime.strptime(text, "%Y-%m-%d %H:%M")` validates a timestamp;
formatting it back with the same pattern lets you require the exact written format.
Test malformed rows, `light` as an unknown sensor, empty text, `nan`, `inf`, a valid
`temp=-10`, and boundary values before cleaning the full file.

### Worked reasoning

Validate structure before indexing, then strip whitespace. Check required fields, known sensor and exact timestamp format. Convert the reading; conversion alone does not reject nan/inf, so check finiteness next. Finally compare with the sensor’s inclusive range. Return a two-item tuple on every branch so callers can always unpack decision and reason.

**Türkçe:** Kontrol sırası indeks hatalarını ve NaN tuzağını önler. Geçerli negatif sıcaklıkları, geçersiz negatif nemden ayırın.

**Expected check:** The five sample cases return OK, empty, non-numeric, out-of-range, out-of-range; edge tests pass.


In [3]:
def valid_timestamp(text):
    try:
        parsed = datetime.strptime(text, TIME_FORMAT)
    except (ValueError, TypeError):
        return False
    return parsed.strftime(TIME_FORMAT) == text

def validate_row(row):
    if not isinstance(row, (list, tuple)) or len(row) != 3:
        return False, "expected exactly 3 fields"
    if not all(isinstance(field, str) for field in row):
        return False, "all fields must be text"
    timestamp, sensor, text = [field.strip() for field in row]
    if not timestamp:
        return False, "empty timestamp"
    if not sensor:
        return False, "empty sensor"
    if not text:
        return False, "empty value"
    if sensor not in SENSOR_RANGES:
        return False, f"unknown sensor: {sensor}"
    if not valid_timestamp(timestamp):
        return False, "invalid timestamp (expected YYYY-MM-DD HH:MM)"
    try:
        value = float(text)
    except ValueError:
        return False, f"non-numeric value: {text!r}"
    if not math.isfinite(value):
        return False, f"nonfinite value: {text}"
    lower, upper = SENSOR_RANGES[sensor]
    if not lower <= value <= upper:
        return False, f"out of range: {value}"
    return True, "OK"

for row in [raw_data[0], raw_data[5], raw_data[8], raw_data[4], raw_data[13]]:
    print(row, "->", validate_row(row))
time = "2024-01-15 08:00"
assert validate_row([time, "temp", "-10"]) == (True, "OK")
for sensor, (lower, upper) in SENSOR_RANGES.items():
    assert validate_row([time, sensor, str(lower)])[0]
    assert validate_row([time, sensor, str(upper)])[0]
for row in [[], [time,"temp"], [time,"temp","2","extra"],
            [time,"light","1"], [time,"temp","nan"], [time,"temp","inf"],
            [time,"temp","-inf"], ["bad-date","temp","2"], [time,"temp",2],
            [time,"humidity","-1"], [time,"pressure","1201"]]:
    assert not validate_row(row)[0], row
print("Row validation checks passed")


['2024-01-15 08:00', 'temp', '22.5'] -> (True, 'OK')
['2024-01-15 08:25', 'humidity', ''] -> (False, 'empty value')
['2024-01-15 08:40', 'pressure', 'abc'] -> (False, "non-numeric value: 'abc'")
['2024-01-15 08:20', 'temp', '-999'] -> (False, 'out of range: -999.0')
['2024-01-15 09:05', 'humidity', '150.0'] -> (False, 'out of range: 150.0')
Row validation checks passed


## Exercise 4: Clean the Data

Using your `validate_row()` function, separate the data into two lists:
- `clean_data` — rows that passed validation
- `bad_data` — rows that failed, along with the reason

Print a summary of the cleaning results.

**Expected output:**
```
Data Cleaning Results
=====================
Total rows: 20
Clean rows: 15
Bad rows: 5

Bad rows detail:
  Row 5: 2024-01-15 08:20 | temp | -999 -> out of range: -999.0
  Row 6: 2024-01-15 08:25 | humidity |  -> empty value
  Row 9: 2024-01-15 08:40 | pressure | abc -> non-numeric value: 'abc'
  Row 14: 2024-01-15 09:05 | humidity | 150.0 -> out of range: 150.0
  Row 19: 2024-01-15 09:30 | temp |  -> empty value
```

### Worked reasoning

Keep the validator focused on one row. The cleaning function loops over all rows, numbers them from 1, and appends each to either clean data or rejection details. Stripping accepted fields creates normalized rows without changing the supplied raw data. Counts must satisfy clean+bad=raw.

**Türkçe:** Temizleme döngüsü karar üretmez; tek satır doğrulayıcısının kararını kullanıp veriyi iki listeye ayırır.

**Expected check:** 20 total, 15 clean, 5 bad; rejected record numbers5,6,9,14,19.


In [4]:
def clean_records(rows):
    clean, bad = [], []
    for number, row in enumerate(rows, start=1):
        valid, reason = validate_row(row)
        if valid:
            clean.append([field.strip() for field in row])
        else:
            bad.append((number, row, reason))
    return clean, bad

clean_data, bad_data = clean_records(raw_data)
print("Data Cleaning Results\n=====================")
print(f"Total rows: {len(raw_data)}\nClean rows: {len(clean_data)}\nBad rows: {len(bad_data)}")
print("\nBad rows detail:")
for number, row, reason in bad_data:
    print(f"  Row {number}: {' | '.join(row)} -> {reason}")
assert (len(raw_data), len(clean_data), len(bad_data)) == (20, 15, 5)
assert [number for number, _, _ in bad_data] == [5, 6, 9, 14, 19]
assert all(validate_row(row)[0] for row in clean_data)
assert clean_records([]) == ([], [])


Data Cleaning Results
Total rows: 20
Clean rows: 15
Bad rows: 5

Bad rows detail:
  Row 5: 2024-01-15 08:20 | temp | -999 -> out of range: -999.0
  Row 6: 2024-01-15 08:25 | humidity |  -> empty value
  Row 9: 2024-01-15 08:40 | pressure | abc -> non-numeric value: 'abc'
  Row 14: 2024-01-15 09:05 | humidity | 150.0 -> out of range: 150.0
  Row 19: 2024-01-15 09:30 | temp |  -> empty value


## Exercise 5: Stats for One Sensor

Write a function called `compute_stats(data, sensor_name)` that takes the clean data list and a sensor name, and returns a dictionary with `count`, `min`, `max`, and `mean` for that sensor.

**Expected output (test):**
```
Stats for temp:
  count: 6
  min: 22.2
  max: 24.0
  mean: 23.02
```



**Hand check:** valid temperature values are `22.5, 23.1, 22.8, 23.5, 24.0, 22.2`; sum `138.1`, count `6`, mean `138.1/6 = 23.0166…`, displayed `23.02`. Humidity has `5` readings with mean `45.38`; pressure has `4` with mean `1012.8625`. Preserve precision internally.

### Worked reasoning

Filter by sensor name before arithmetic so different units are never mixed. Temperature values sum to 138.1; count 6 gives mean 23.0166…, displayed 23.02. Store full precision. For a known sensor with no readings, return count 0 with None statistics; an unknown sensor name is a caller error.

**Türkçe:** Aynı sensöre ait ölçümleri toplayın. Veri yoksa ortalama sıfır değil, tanımsızdır.

**Expected check:** Temperature count 6, min22.2, max24.0, mean 23.02; empty group returns None statistics.


In [5]:
def compute_stats(data, sensor_name):
    if sensor_name not in SENSOR_RANGES:
        raise ValueError(f"unknown sensor: {sensor_name}")
    values = []
    for row in data:  # contract: data already passed clean_records
        if row[1] == sensor_name:
            values.append(float(row[2]))
    if not values:
        return {"count": 0, "min": None, "max": None, "mean": None}
    return {"count": len(values), "min": min(values), "max": max(values),
            "mean": sum(values) / len(values)}

temp_stats = compute_stats(clean_data, "temp")
print("Stats for temp:")
print("  count:", temp_stats["count"])
print("  min:", temp_stats["min"])
print("  max:", temp_stats["max"])
print(f"  mean: {temp_stats['mean']:.2f}")
assert temp_stats["count"] == 6
assert (temp_stats["min"], temp_stats["max"]) == (22.2,24.0)
assert math.isclose(temp_stats["mean"], 138.1 / 6)
assert compute_stats([], "temp") == {"count":0,"min":None,"max":None,"mean":None}
try:
    compute_stats(clean_data, "light")
except ValueError:
    pass
else:
    raise AssertionError("unknown sensor should fail")


Stats for temp:
  count: 6
  min: 22.2
  max: 24.0
  mean: 23.02


## Exercise 6: Stats for All Sensors

Use your `compute_stats()` function to compute statistics for **all three sensors** (`temp`, `humidity`, `pressure`). Store the results in a dictionary called `all_stats`.

**Expected output:**
```
Sensor Statistics
=================

temp:
  Readings: 6
  Min: 22.20, Max: 24.00, Mean: 23.02

humidity:
  Readings: 5
  Min: 43.50, Max: 47.30, Mean: 45.38

pressure:
  Readings: 4
  Min: 1011.90, Max: 1013.50, Mean: 1012.86
```

### Worked reasoning

Use the same function for every sensor, storing each result under its sensor key. Temperature count 6 plus humidity5 plus pressure4 equals15 clean rows. Means are138.1/6,226.9/5 and4051.45/4. Their units differ; the report presents them separately.

**Türkçe:** Bir fonksiyonu üç sensörde tekrar kullanmak, üç ayrı ve tutarsız hesap yazmayı önler.

**Expected check:** Temperature6/23.02, humidity5/45.38, pressure4/1012.86 (count/mean).


In [6]:
def stats_for_all(data):
    results = {}
    for sensor in SENSOR_RANGES:
        results[sensor] = compute_stats(data, sensor)
    return results

all_stats = stats_for_all(clean_data)
print("Sensor Statistics\n=================")
for sensor, stats in all_stats.items():
    print(f"\n{sensor}:\n  Readings: {stats['count']}")
    if stats["count"]:
        print(f"  Min: {stats['min']:.2f}, Max: {stats['max']:.2f}, Mean: {stats['mean']:.2f}")
    else:
        print("  Min: N/A, Max: N/A, Mean: N/A")
assert [stats["count"] for stats in all_stats.values()] == [6,5,4]
assert sum(stats["count"] for stats in all_stats.values()) == len(clean_data)
assert math.isclose(all_stats["humidity"]["mean"], 45.38)
assert math.isclose(all_stats["pressure"]["mean"], 1012.8625)


Sensor Statistics

temp:
  Readings: 6
  Min: 22.20, Max: 24.00, Mean: 23.02

humidity:
  Readings: 5
  Min: 43.50, Max: 47.30, Mean: 45.38

pressure:
  Readings: 4
  Min: 1011.90, Max: 1013.50, Mean: 1012.86


## Exercise 7: Generate Summary Report

Write a function called `generate_report(all_stats, total_rows, clean_count, bad_count)` that returns a formatted string containing the full summary report.

The report should look like this:
```
======================================
    SENSOR LOG SUMMARY REPORT
======================================

DATA OVERVIEW
-------------
Total readings:   20
Valid readings:   15
Invalid readings: 5
Data quality:     75.0%

SENSOR: temp
-------------
  Readings: 6
  Min:      22.20
  Max:      24.00
  Mean:     23.02

SENSOR: humidity
-------------
  Readings: 5
  Min:      43.50
  Max:      47.30
  Mean:     45.38

SENSOR: pressure
-------------
  Readings: 4
  Min:      1011.90
  Max:      1013.50
  Mean:     1012.86

======================================
```



**Empty-group formatting:** render `None` as `N/A`. If total rows are zero, report data quality as `N/A` rather than dividing by zero. Check that sensor counts sum to the clean-row count.

### Worked reasoning

Separate numeric work from display. Build a list of report lines and join them with newlines. Validate that the reported totals agree, format finite measurements to two decimals and None as N/A. Data quality is15/20×100=75%; with no input records the ratio is undefined and displayed N/A.

**Türkçe:** Rapor biçimlendirmesi hesabı değiştirmez. Toplamlar uyuşmuyorsa güzel görünen yanlış bir rapor üretmeyin.

**Expected check:** A coherent20/15/5 report with75.0% quality and the corrected per-sensor statistics.


In [7]:
def format_stat(value):
    return "N/A" if value is None else f"{value:.2f}"

def generate_report(all_stats, total_rows, clean_count, bad_count):
    if total_rows != clean_count + bad_count:
        raise ValueError("total must equal clean plus rejected")
    if sum(stats["count"] for stats in all_stats.values()) != clean_count:
        raise ValueError("sensor counts must equal the clean-row count")
    quality = f"{100 * clean_count / total_rows:.1f}%" if total_rows else "N/A"
    lines = ["======================================", "    SENSOR LOG SUMMARY REPORT",
             "======================================", "", "DATA OVERVIEW", "-------------",
             f"Total readings:   {total_rows}", f"Valid readings:   {clean_count}",
             f"Invalid readings: {bad_count}", f"Data quality:     {quality}"]
    units = {"temp":"°C", "humidity":"%", "pressure":"hPa"}
    for sensor, stats in all_stats.items():
        lines.extend(["", f"SENSOR: {sensor} ({units[sensor]})", "-------------",
                      f"  Readings: {stats['count']}", f"  Min:      {format_stat(stats['min'])}",
                      f"  Max:      {format_stat(stats['max'])}", f"  Mean:     {format_stat(stats['mean'])}"])
    return "\n".join(lines) + "\n\n======================================\n"

report = generate_report(all_stats, len(raw_data), len(clean_data), len(bad_data))
print(report)
assert "Valid readings:   15" in report and "75.0%" in report
assert "23.02" in report and "45.38" in report and "1012.86" in report
empty_report = generate_report(stats_for_all([]), 0, 0, 0)
assert "Data quality:     N/A" in empty_report
assert empty_report.count("Mean:     N/A") == 3


    SENSOR LOG SUMMARY REPORT

DATA OVERVIEW
-------------
Total readings:   20
Valid readings:   15
Invalid readings: 5
Data quality:     75.0%

SENSOR: temp (°C)
-------------
  Readings: 6
  Min:      22.20
  Max:      24.00
  Mean:     23.02

SENSOR: humidity (%)
-------------
  Readings: 5
  Min:      43.50
  Max:      47.30
  Mean:     45.38

SENSOR: pressure (hPa)
-------------
  Readings: 4
  Min:      1011.90
  Max:      1013.50
  Mean:     1012.86




## Exercise 8: Write Clean CSV

Write the `clean_data` to a file called `"sensor_data_clean.csv"`. Include the header row. Then read the file back and print the first 5 lines to verify.

**Expected output:**
```
Clean data written to sensor_data_clean.csv (15 rows + header)

First 5 lines of clean file:
  timestamp,sensor,value
  2024-01-15 08:00,temp,22.5
  2024-01-15 08:05,humidity,45.2
  2024-01-15 08:10,temp,23.1
  2024-01-15 08:15,pressure,1013.25
```

### Worked reasoning

Persist the validated rows with a single header. csv.writer handles field quoting and newline rules; read the file back with csv.reader to compare every field. The result must have16 physical rows for this dataset: one header plus15 valid records.

**Türkçe:** Dosyayı yeniden okuyup tüm alanları karşılaştırın; yalnızca ekrandaki başarı mesajına güvenmeyin.

**Expected check:** The CSV contains the exact15 validated rows plus the required header.


In [8]:
def write_clean_csv(data, filename="sensor_data_clean.csv"):
    with open(filename, "w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file, lineterminator="\n")
        writer.writerow(HEADER)
        writer.writerows(data)

write_clean_csv(clean_data)
with open("sensor_data_clean.csv", "r", encoding="utf-8", newline="") as file:
    saved = list(csv.reader(file))
print(f"Clean data written to sensor_data_clean.csv ({len(clean_data)} rows + header)")
print("\nFirst 5 lines of clean file:")
for row in saved[:5]:
    print("  " + ",".join(row))
assert saved[0] == HEADER
assert saved[1:] == clean_data and len(saved) == 16


Clean data written to sensor_data_clean.csv (15 rows + header)

First 5 lines of clean file:
  timestamp,sensor,value
  2024-01-15 08:00,temp,22.5
  2024-01-15 08:05,humidity,45.2
  2024-01-15 08:10,temp,23.1
  2024-01-15 08:15,pressure,1013.25


## Exercise 9: Write Report to File

Write the report string (from Exercise 7) to a file called `"sensor_report.txt"`. Print a confirmation and then read back the file to verify.

**Expected output:**
```
Report written to sensor_report.txt

--- sensor_report.txt ---
======================================
    SENSOR LOG SUMMARY REPORT
======================================
...
```

### Worked reasoning

Write the report string already computed in EX7, then reopen it and compare the entire content. This is a required project deliverable, not an optional formatting exercise. Keep UTF-8 so the units and Turkish names remain readable.

**Türkçe:** Metni üretmek ile dosyaya kaydetmek ayrı aşamalardır; ikisi de proje için gereklidir.

**Expected check:** sensor_report.txt exists, and its contents equal the generated report.


In [9]:
def write_report(report, filename="sensor_report.txt"):
    with open(filename, "w", encoding="utf-8") as file:
        file.write(report)

write_report(report)
with open("sensor_report.txt", "r", encoding="utf-8") as file:
    saved_report = file.read()
print("Report written to sensor_report.txt")
print("--- report read back ---")
print(saved_report)
assert saved_report == report
assert "SENSOR: pressure (hPa)" in saved_report


Report written to sensor_report.txt
--- report read back ---
    SENSOR LOG SUMMARY REPORT

DATA OVERVIEW
-------------
Total readings:   20
Valid readings:   15
Invalid readings: 5
Data quality:     75.0%

SENSOR: temp (°C)
-------------
  Readings: 6
  Min:      22.20
  Max:      24.00
  Mean:     23.02

SENSOR: humidity (%)
-------------
  Readings: 5
  Min:      43.50
  Max:      47.30
  Mean:     45.38

SENSOR: pressure (hPa)
-------------
  Readings: 4
  Min:      1011.90
  Max:      1013.50
  Mean:     1012.86




## Exercise 10: The Complete Program

Write a `main()` function that does everything:
1. Reads `sensor_data.csv`
2. Validates and cleans the data
3. Computes statistics for each sensor
4. Generates a summary report
5. Writes `sensor_data_clean.csv`
6. Writes `sensor_report.txt`
7. Prints a final status message

Combine all your functions from previous exercises. Call `main()` at the end.

**Expected output:**
```
Sensor Log Summary Tool
=======================

[1/6] Reading sensor_data.csv...
      Found 20 data rows.

[2/6] Cleaning data...
      Valid: 15 | Invalid: 5

[3/6] Computing statistics...
      Processed 3 sensors.

[4/6] Generating report...
      Report ready.

[5/6] Writing sensor_data_clean.csv...
      Done.

[6/6] Writing sensor_report.txt...
      Done.

All tasks completed successfully!
```



**Required robustness checks:** run once on the supplied 20-row dataset, then test an empty file, header-only file, wrong header, malformed rows, unknown sensors, invalid timestamps, nonfinite numbers, and valid negative temperatures. Each bad record receives a reason; it does not stop later valid records. Verify the persisted files by reading them back.

### Worked reasoning

Compose the tested stages without copying their logic. File-level errors return a clear failed result; bad records are handled by clean_records and do not abort later records. Empty input still succeeds and writes a header-only CSV plus an N/A report. Tests use temporary fixture folders so the canonical 20-record outputs remain available after the notebook runs.

**Türkçe:** Ana fonksiyon aşamaları birleştirir. Tek bozuk kayıt tüm dosyayı durdurmaz; dosya düzeyindeki hata ise açıkça bildirilir.

**Expected check:** Canonical outputs contain 15 valid records and corrected statistics; all listed robustness cases pass.


In [10]:
def main(filepath="sensor_data.csv", output_dir="."):
    try:
        rows = read_data(filepath)
        clean, bad = clean_records(rows)
        statistics = stats_for_all(clean)
        text = generate_report(statistics, len(rows), len(clean), len(bad))
        directory = Path(output_dir)
        directory.mkdir(parents=True, exist_ok=True)
        clean_path = directory / "sensor_data_clean.csv"
        report_path = directory / "sensor_report.txt"
        write_clean_csv(clean, clean_path)
        write_report(text, report_path)
    except (OSError, ValueError, csv.Error) as error:
        return {"ok": False, "error": f"{type(error).__name__}: {error}"}
    return {"ok": True, "total": len(rows), "valid": len(clean), "rejected": len(bad),
            "stats": statistics, "errors": bad, "clean_path": clean_path, "report_path": report_path}

result = main()
assert result["ok"]
print("Sensor Log Summary Tool")
print(f"Total: {result['total']} | Valid: {result['valid']} | Rejected: {result['rejected']}")
print("Saved:", result["clean_path"], "and", result["report_path"])
assert (result["total"], result["valid"], result["rejected"]) == (20,15,5)
assert read_data(result["clean_path"]) == clean_data

from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="cp1_w14_checks_", dir=".") as folder:
    base = Path(folder)
    for name, contents in [("empty", ""), ("header_only", "timestamp,sensor,value\n")]:
        input_path = base / f"{name}.csv"
        input_path.write_text(contents, encoding="utf-8")
        checked = main(input_path, base / name)
        assert checked["ok"] and checked["valid"] == checked["total"] == 0
        assert read_data(checked["clean_path"]) == []
        text = checked["report_path"].read_text(encoding="utf-8")
        assert text.count("Mean:     N/A") == 3

    # One valid negative temperature survives eleven independent bad records.
    invalid_cases = [
        [time,"temp"], [time,"temp","2","extra"],
        [time,"light","12"], ["bad-date","temp","10"],
        [time,"temp","nan"], [time,"temp","inf"], [time,"temp","-inf"],
        ["","temp","10"], [time,"temp",""], [time,"temp","abc"],
        [time,"humidity","101"], [time,"temp","-10"],
    ]
    mixed_path = base / "mixed.csv"
    with open(mixed_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(HEADER)
        writer.writerows(invalid_cases)
    checked = main(mixed_path, base / "mixed")
    assert checked["ok"] and (checked["total"],checked["valid"],checked["rejected"]) == (12,1,11)
    assert checked["stats"]["temp"]["mean"] == -10
    assert checked["stats"]["humidity"]["mean"] is None
    assert read_data(checked["clean_path"]) == [[time,"temp","-10"]]

    bad_header = base / "bad_header.csv"
    bad_header.write_text("wrong,header\n", encoding="utf-8")
    assert not main(bad_header, base / "bad_header")["ok"]
    missing = main(base / "not_present.csv", base / "missing")
    assert not missing["ok"] and "FileNotFoundError" in missing["error"]

print("Pipeline checks passed: reference, empty, header-only, malformed, unknown sensor,")
print("timestamp, nonfinite, range, valid negative temperature, wrong header, missing file.")


Sensor Log Summary Tool
Total: 20 | Valid: 15 | Rejected: 5
Saved: sensor_data_clean.csv and sensor_report.txt
Pipeline checks passed: reference, empty, header-only, malformed, unknown sensor,
timestamp, nonfinite, range, valid negative temperature, wrong header, missing file.


## Exercise 11: BONUS — Date/Time Filtering (Challenge)

Add a feature to your program that filters the **cleaned data** by a time range. Write a function called `filter_by_time(data, start_time, end_time)` that returns only rows where the timestamp is between `start_time` and `end_time` (inclusive).

Since timestamps are in `YYYY-MM-DD HH:MM` format, you can compare them as strings (alphabetical comparison works correctly for this format).

**Expected output:**
```
Filtering: 2024-01-15 08:30 to 2024-01-15 09:00
Found 6 readings in time range.

Filtered data:
  2024-01-15 08:30 | temp      | 22.8
  2024-01-15 08:35 | humidity  | 46.1
  ...
```


The 08:40 pressure record contains `abc` and is rejected before filtering. The six retained timestamps are 08:30, 08:35, 08:45, 08:50, 08:55, and 09:00. Validate the endpoints and require start ≤ end.

### Worked reasoning

Filter clean_data, never the raw input. The inclusive08:30–09:00 interval has seven raw timestamps but one pressure row contains abc and was already rejected. Six validated readings remain. Exact ISO-style timestamps sort chronologically as strings; validate both endpoints before comparing.

**Türkçe:** Önce temizleme, sonra zaman filtresi. Yedi ham satırın biri bozuk olduğu için sonuç altı satırdır.

**Expected check:** Six cleaned readings, including both endpoints; the invalid08:40 pressure record is absent.


In [11]:
def filter_by_time(data, start_time, end_time):
    if not valid_timestamp(start_time) or not valid_timestamp(end_time):
        raise ValueError("invalid time-range endpoint")
    if start_time > end_time:
        raise ValueError("start_time must not exceed end_time")
    selected = []
    for row in data:  # contract: already validated and cleaned
        if start_time <= row[0] <= end_time:
            selected.append(row)
    return selected

start = "2024-01-15 08:30"
end = "2024-01-15 09:00"
filtered = filter_by_time(clean_data, start, end)
print(f"Filtering: {start} to {end}\nFound {len(filtered)} readings in time range.")
for timestamp, sensor, value in filtered:
    print(f"  {timestamp} | {sensor:<9} | {value}")
assert len(filtered) == 6
assert [row[0][-5:] for row in filtered] == ["08:30","08:35","08:45","08:50","08:55","09:00"]
assert len(filter_by_time(clean_data, start, start)) == 1
assert filter_by_time([], start, end) == []
assert all(validate_row(row)[0] for row in filtered)
try:
    filter_by_time(clean_data, end, start)
except ValueError:
    pass
else:
    raise AssertionError("reversed time interval should fail")


Filtering: 2024-01-15 08:30 to 2024-01-15 09:00
Found 6 readings in time range.
  2024-01-15 08:30 | temp      | 22.8
  2024-01-15 08:35 | humidity  | 46.1
  2024-01-15 08:45 | temp      | 23.5
  2024-01-15 08:50 | humidity  | 44.8
  2024-01-15 08:55 | pressure  | 1012.80
  2024-01-15 09:00 | temp      | 24.0


## Exercise 12: BONUS — Sensor Comparison (Challenge)

Write a function called `compare_sensors(all_stats)` that prints a comparison table showing all sensors side by side.

**Expected output:**
```
Sensor Comparison Table
=======================

Metric       temp        humidity    pressure   
------       ----        --------    --------   
Count        6           5           4          
Min          22.20       43.50       1011.90    
Max          24.00       47.30       1013.50    
Mean         23.02       45.38       1012.86    
Range        1.80        3.80        1.60       
```

The **range** is `max - min`.



**Interpretation:** compare counts and processing quality across sensors, but do not rank temperature, humidity, and pressure means as if they had the same unit. Display N/A for an empty sensor group.

### Worked reasoning

Read each sensor’s dictionary and align columns. Range is max−min: temperature1.8°C, humidity3.8 percentage points, pressure1.6hPa. Handle None before subtraction. This is a presentation comparison, not a ranking of measurements with incompatible units.

**Türkçe:** Aralık aynı sensörün maksimumundan minimumunu çıkarır. Farklı birimlerdeki sensör ortalamalarını büyüklük yarışına çevirmeyin.

**Expected check:** Counts6/5/4, means23.02/45.38/1012.86 and ranges1.80/3.80/1.60, with units and empty handling.


In [12]:
def compare_sensors(statistics):
    sensors = list(SENSOR_RANGES)
    labels = {"temp":"temp °C", "humidity":"humidity %", "pressure":"pressure hPa"}
    lines = ["Sensor Comparison Table", "=" * 52,
             f"{'Metric':<12}" + "".join(f"{labels[sensor]:>14}" for sensor in sensors)]
    for metric in ["count", "min", "max", "mean", "range"]:
        cells = []
        for sensor in sensors:
            stats = statistics[sensor]
            if metric == "range":
                value = None if stats["count"] == 0 else stats["max"] - stats["min"]
            else:
                value = stats[metric]
            text = str(value) if metric == "count" else format_stat(value)
            cells.append(f"{text:>14}")
        lines.append(f"{metric.title():<12}" + "".join(cells))
    table = "\n".join(lines)
    print(table)
    return table

comparison = compare_sensors(all_stats)
assert "23.02" in comparison and "45.38" in comparison and "1012.86" in comparison
assert [stats["count"] for stats in all_stats.values()] == [6,5,4]
assert [round(stats["max"]-stats["min"],2) for stats in all_stats.values()] == [1.8,3.8,1.6]
empty_comparison = compare_sensors(stats_for_all([]))
assert empty_comparison.count("N/A") == 12


Sensor Comparison Table
Metric             temp °C    humidity %  pressure hPa
Count                    6             5             4
Min                  22.20         43.50       1011.90
Max                  24.00         47.30       1013.50
Mean                 23.02         45.38       1012.86
Range                 1.80          3.80          1.60
Sensor Comparison Table
Metric             temp °C    humidity %  pressure hPa
Count                    0             0             0
Min                    N/A           N/A           N/A
Max                    N/A           N/A           N/A
Mean                   N/A           N/A           N/A
Range                  N/A           N/A           N/A


## Review before moving on

Choose one solution, change one input, and predict the new result before rerunning it.
Explain which branch, loop invariant, validation rule, or function contract makes the
answer correct. A familiar answer copied unchanged is weaker evidence than a new case
you can explain.

**Türkçe:** Bir girdiyi değiştirin, çalıştırmadan sonucu tahmin edin ve farkı açıklayın.
